## Metadata spreadsheet batch converter

Purpose: This notebook pulls API information into a tab in a spreadsheet and coverts the spreadsheet into json files for upload into the DDE

Rationale: It has been deemed desirable to generate ResourceCatalog records for Sources. To minimize additional curation, some of the content is generated based on aggregations from the ingested records and stored on the API. For the sake of consistency, this spreadsheet will pull that information from the API

The tabbed sheets available in the spreadsheet are as follows:
- resource_base (available)
- ~funding~
- collectionSize (to be generated)
- ~related~
- spatialTemporalcoverage (to be generated)
- ~author~
- definedTerms (to be generated)
- ~distribution~

All sheets use the url field as the index/linking field

Notes:
* resource_base: contains various metadata properties and their expected values. For citation.pmid, a helper function should be used to pull the citation name based on the pmid so that it will be able to pass the schema validation
* funding: Note a single grant ID may be associated with multiple funding organizations. Convert 'type' to '@type'
* related: relationship properties and their expected objects. Convert 'type' to '@type'
* spatialTemporalcoverage: maps rows to `spatialCoverage` (`AdministrativeArea`) and `temporalCoverage` (`TemporalInterval`) objects following the NDE ResourceCatalog schema
* definedTerm: save only the urls to a list for the DDE

How it works: 
Every sheet except for the resource_base is converted into a dictionary where the key is the url, and the value is either an array of objects (funding, collectionSize, author) or a dictionary with additional objects (related, definedTerm)
The resource_base is converted into a base dictionary, and additional objects are added to the base dictionary using the url
The json records are then dumped into a batch_file for upload


In [1]:
import os
import pandas as pd
import json
from datetime import datetime
from Bio import Entrez
from Bio import Medline
import requests
from math import isnan
import time
from openpyxl import load_workbook

In [2]:
script_path = os.getcwd()
parent_path = os.path.abspath(os.path.join(script_path, os.pardir))
data_path = os.path.join(script_path,'data')
filelist = os.listdir(data_path)
result_path = os.path.join(parent_path,'nde-metadata-corrections','metadata_for_DDE','resourceCatalogs')

In [3]:
print(parent_path)
print(filelist)

C:\Users\gtsueng\Anaconda3\envs\nde
['2024_05_14_RepoMetaCuration.xlsx', '2024_05_21_RepoMetaCuration.xlsx', '2024_06_18_RepoMetaCuration.xlsx', '2024_06_21_RepoMetaCuration.xlsx', '2024_06_28_RepoMetaCuration.xlsx', '2024_07_20_RepoMetaCuration.xlsx', '2024_07_26_RepoMetaCuration.xlsx', '2024_07_31_RepoMetaCuration.xlsx', '2024_12_20_RepoMetaCuration.xlsx', '2025_01_23_RepoMetaCuration.xlsx', '2025_03_19_RepoMetaCuration.xlsx', '2025_03_25_RepoMetaCuration.xlsx', '2025_04_18_RepoMetaCuration.xlsx', '2025_04_24_RepoMetaCuration.xlsx', '2025_04_25_RepoMetaCuration.xlsx', '2025_05_14_RepoMetaCuration.xlsx', '2025_09_29_RepoMetaCuration.xlsx', '2025_09_30_RepoMetaCuration.xlsx', '2025_11_25_RepoMetaCuration.xlsx', '2025_12_04_RepoMetaCuration.xlsx', '2026_04_08_RepoMetaCuration.xlsx', '2026_04_13_RepoMetaCuration.xlsx', '2026_04_14_RepoMetaCuration_ingested_sources.xlsx', '2026_06_16_RepoMetaCuration.xlsx', '2026_06_25_SourceMetaCuration.xlsx']


In [4]:
def clean_nones(a_dict):
    for k,v in list(a_dict.items()):
        if v == -1:
            del a_dict[k]
        if v == "None":
            del a_dict[k]
        if v == None:
            del a_dict[k]
        if not isinstance(v,str) and not isinstance(v,dict) and not isinstance(v,list):
            if isnan(v):
                del a_dict[k]
    return a_dict


def clean_dict_array(dict_array):
    for eachdict in dict_array:
        eachdict = clean_nones(eachdict)
    return dict_array

### Process the resource_base sheet

Note that the citation/pmid/, license, usageInfo, Language fields will be empty without curation

In [5]:
def add_date(df):
    today = datetime.now()
    df['date'] = today.strftime("%Y-%m-%d")
    return df

def format_date(datefield):
    if isinstance(datefield,str)==True:
        cleandate = datefield
    if isinstance(datefield,datetime)==True:
        cleandate = datefield.strftime("%Y-%m-%d")
    return cleandate

def add_type(df):
    df['@type'] = 'nde:ResourceCatalog'
    return df

def select_sources(df):
    ### drop non-dataset or computational tool sources
    datasetdf = df.loc[((df['collectionType']=='Dataset Repository')|(df['collectionType']=='Computational Tool Repository'))]
    noimmune = datasetdf.loc[datasetdf['name']!="ImmuneSpace"].copy()
    cleandf = noimmune[['name','url','identifier','alternateName','conditionsOfAccess',
                        'abstract','description','collectionType','isAccessibleForFree','genre','creativeWorkStatus']].copy()
    return cleandf

def run_quick_clean(df):
    ### fill the na's
    df = df.fillna("None")
    df['genre'] = df['genre'].astype(str).replace('generalist','Generalist')
    print(df)
    return df

def get_meta(server='prod'):
    if server == 'prod':
        metaurl = 'https://api.data.niaid.nih.gov/v1/metadata'
    else:
        metaurl = 'https://api-staging.data.niaid.nih.gov/v1/metadata'
    
    r = requests.get('https://api.data.niaid.nih.gov/v1/metadata')
    res = json.loads(r.text)
    sourcemeta = res['src']
    print(sourcemeta.keys())
    return sourcemeta

In [6]:
sourcemeta = get_meta('staging')

dict_keys(['zenodo', 'reframedb', 'hca', 'microbiomedb', 'omicsdi', 'immunespace', 'veupathdb', 'vivli', 'hubmap', 'clinepidb', 'qiita', 'ndex', 'ark', 'tycho', 'dataverse', 'lincs', 'malariagen', 'dde', 'ncbi_bioproject', 'ncbi_sra', 'acd_niaid', 'immport', 'pdb', 'biotools', 'mendeley', 'massive', 'figshare', 'veupath_collections', 'dbgap', 'ncbi_geo', 'dryad', 'flowrepository', 'dash', 'covid_radx', 'vdj', 'proteomexchange', 'amoebadb', 'cryptodb', 'fungidb', 'giardiadb', 'hostdb', 'microsporidiadb', 'piroplasmadb', 'plasmodb', 'toxodb', 'trichdb', 'tritrypdb', 'vectorbase'])


In [7]:
filepath = os.path.join(data_path,filelist[-1])
df_base = pd.read_excel(filepath, 'resource_base', engine='openpyxl')
df_subset = select_sources(df_base)
df_clean = add_type(run_quick_clean(df_subset))
print(df_clean.head(n=2))
#check = df_clean.iloc[0]['url']

                                                 name  \
0                            accessclinicaldata@NIAID   
1                                           ClinEpiDB   
2                                 COVID RADx Data Hub   
3   ImmPort - Bioinformatics For the Future of Imm...   
4                                          MalariaGEN   
5                                             MassIVE   
6                                            NCBI GEO   
7                  NICHD Data and Specimen Hub (DASH)   
8                                   Protein Data Bank   
9                                               Qiita   
10           The Database of Genotypes and Phenotypes   
11                   The Network Data Exchange (NDEx)   
12                                          VDJServer   
13                                           AmoebaDB   
14                                          bio.tools   
15                                           CryptoDB   
16                             

In [ ]:
check = df_clean.iloc[0]['url']
print(check)

### Generate the collectionSize sheet

In [8]:
def fetch_collectionSize(df_clean,sourcemeta):
    srcids = df_clean['identifier'].unique().tolist()
    srcnamelist = list(sourcemeta.keys())
    collectionlist = []
    for eachsrc in srcnamelist:
        try:
            if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
                collobject = sourcemeta[eachsrc]['sourceInfo']['collectionSize']
                if isinstance(collobject, dict): 
                    collection = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                  'minValue':collobject['minValue'],
                                  'unitText': collobject['unitText']}
                    collectionlist.append(collection)
                elif isinstance(collobject, list):
                    i=0
                    while i < len(collobject):
                        collection = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                      'minValue':collobject[i]['minValue'],
                                      'unitText': collobject[i]['unitText']}               
                        collectionlist.append(collection)
                        i = i+1
        except:
            print(eachsrc," has no sourceInfo")

    collectiondf = pd.DataFrame(collectionlist)
    return collectiondf

In [ ]:
collectiondf = fetch_collectionSize(df_clean, sourcemeta)

### Generate the definedTerms sheet

For each source meta record, query the API, pull the DefinedTerm urls, and add them to a DataFrame for export as a sheet

In [9]:
def fetch_definedterms(df_clean, sourcemeta):
    srcids = df_clean['identifier'].unique().tolist()
    srcnamelist = list(sourcemeta.keys())
    dttypes = ['topicCategory','healthCondition','species','infectiousAgent','measurementTechnique']
    definedtermlist = []
    for eachsrc in srcnamelist:
        try:
            if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
                sourcekeys = list(sourcemeta[eachsrc]['sourceInfo'].keys())
                dttypesincommon = set(dttypes).intersection(set(sourcekeys))
                for eachtype in dttypesincommon:
                    dtobject = sourcemeta[eachsrc]['sourceInfo'][eachtype]
                    if isinstance(dtobject, dict): 
                        definedterm = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                      'property': eachtype,
                                      'name': dtobject['name']}
                        try:
                            definedterm['prop.url'] = dtobject['url']
                            definedtermlist.append(definedterm)
                        except:
                            definedtermlist.append(definedterm)               
                    elif isinstance(dtobject, list):
                        i=0
                        while i < len(dtobject):
                            definedterm = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                          'property': eachtype,
                                          'name': dtobject[i]['name']}
                            try:
                                definedterm['prop.url'] = dtobject[i]['url']
                                definedtermlist.append(definedterm)
                            except:
                                definedtermlist.append(definedterm)
                            i = i+1   
        except:
            print(eachsrc," has no sourceInfo")

    definedtermdf = pd.DataFrame(definedtermlist)
    return definedtermdf

In [ ]:
definedtermdf = fetch_definedterms(df_clean, sourcemeta)
print(definedtermdf.head(n=2))

### Generate the spatialTemporalcoverage sheet

In [10]:
def fetch_temporalCoverage(df_clean, sourcemeta):
    srcids = df_clean['identifier'].unique().tolist()
    srcnamelist = list(sourcemeta.keys())
    templist = []
    for eachsrc in srcnamelist:
        try:
            if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
                if 'temporalCoverage' in list(sourcemeta[eachsrc]['sourceInfo'].keys()):
                    tempobject = sourcemeta[eachsrc]['sourceInfo']['temporalCoverage']
                    if isinstance(tempobject, dict): 
                        tempdict = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                      'startDate': tempobject['startDate'],
                                      'temporalType': tempobject['temporalType']}
                        try:
                            tempdict['endDate']= tempobject['endDate']
                            templist.append(tempdict)
                        except:
                            templist.append(tempdict)
                    elif isinstance(tempobject, list):
                        i=0
                        while i < len(tempobject):
                            tempdict = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                          'temporalType': tempobject[i]['temporalType']}
                            try:
                                tempdict['startDate']= tempobject[i]['startDate']
                            except:
                                print(eachsrc,' has no startDate')
                            try:
                                tempdict['endDate']= tempobject[i]['endDate']
                                templist.append(tempdict)
                            except:
                                templist.append(tempdict)
                            i = i+1
        except:
            print(eachsrc," has no temporalCoverage")
    tempdf = pd.DataFrame(templist)
    return tempdf

In [ ]:
tempdf = fetch_temporalCoverage(df_clean, sourcemeta)
print(tempdf.head(n=2))

### Assemble the excel records

In [11]:
def process_source_file(filepath):
    df_base = pd.read_excel(filepath, 'resource_base', engine='openpyxl')
    sourcemeta = get_meta(server='staging')
    df_subset = select_sources(df_base)
    df_clean = add_type(run_quick_clean(df_subset))
    collectiondf = fetch_collectionSize(df_clean, sourcemeta)
    definedtermdf = fetch_definedterms(df_clean, sourcemeta)
    tempdf = fetch_temporalCoverage(df_clean, sourcemeta)
    with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:  
        collectiondf.to_excel(writer, sheet_name='collectionSize', index=False)
        definedtermdf.to_excel(writer, sheet_name='definedTerms', index=False)
        tempdf.to_excel(writer, sheet_name='spatialTemporalCoverage', index=False)    
    print('source file processed')

In [12]:
process_source_file(filepath)

dict_keys(['zenodo', 'reframedb', 'hca', 'microbiomedb', 'omicsdi', 'immunespace', 'veupathdb', 'vivli', 'hubmap', 'clinepidb', 'qiita', 'ndex', 'ark', 'tycho', 'dataverse', 'lincs', 'malariagen', 'dde', 'ncbi_bioproject', 'ncbi_sra', 'acd_niaid', 'immport', 'pdb', 'biotools', 'mendeley', 'massive', 'figshare', 'veupath_collections', 'dbgap', 'ncbi_geo', 'dryad', 'flowrepository', 'dash', 'covid_radx', 'vdj', 'proteomexchange', 'amoebadb', 'cryptodb', 'fungidb', 'giardiadb', 'hostdb', 'microsporidiadb', 'piroplasmadb', 'plasmodb', 'toxodb', 'trichdb', 'tritrypdb', 'vectorbase'])
                                                 name  \
0                            accessclinicaldata@NIAID   
1                                           ClinEpiDB   
2                                 COVID RADx Data Hub   
3   ImmPort - Bioinformatics For the Future of Imm...   
4                                          MalariaGEN   
5                                             MassIVE   
6              

source file processed


### test functions

In [ ]:
r = requests.get('https://api.data.niaid.nih.gov/v1/metadata')
res = json.loads(r.text)
#print(res.keys())
#print(res['src'].keys())
#print(res['src']['reframedb'].keys())
print(res['src']['reframedb']['sourceInfo'].keys())

In [ ]:
srcids = df_clean['identifier'].unique().tolist()
srcnamelist = list(sourcemeta.keys())
collectionlist = []
for eachsrc in srcnamelist:

    if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
        collobject = sourcemeta[eachsrc]['sourceInfo']['collectionSize']
        if isinstance(collobject, dict): 
            collection = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                          'minValue':collobject['minValue'],
                          'unitText': collobject['unitText']}
            collectionlist.append(collection)
        elif isinstance(collobject, list):
            i=0
            while i < len(collobject):
                collection = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                              'minValue':collobject[i]['minValue'],
                              'unitText': collobject[i]['unitText']}               
                collectionlist.append(collection)
                i = i+1

collectiondf = pd.DataFrame(collectionlist)

In [ ]:
srcids = df_clean['identifier'].unique().tolist()
srcnamelist = list(sourcemeta.keys())
dttypes = ['topicCategory','healthCondition','species','infectiousAgent','measurementTechnique']
definedtermlist = []
for eachsrc in srcnamelist:

    if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
        sourcekeys = list(sourcemeta[eachsrc]['sourceInfo'].keys())
        dttypesincommon = set(dttypes).intersection(set(sourcekeys))
        for eachtype in dttypesincommon:
            dtobject = sourcemeta[eachsrc]['sourceInfo'][eachtype]
            if isinstance(dtobject, dict): 
                definedterm = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                              'property': eachtype,
                              'name': dtobject['name']}
                try:
                    definedterm['prop.url'] = dtobject['url']
                    definedtermlist.append(definedterm)
                except:
                    definedtermlist.append(definedterm)               
            elif isinstance(dtobject, list):
                i=0
                while i < len(dtobject):
                    definedterm = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                  'property': eachtype,
                                  'name': dtobject[i]['name']}
                    try:
                        definedterm['prop.url'] = dtobject[i]['url']
                        definedtermlist.append(definedterm)
                    except:
                        definedtermlist.append(definedterm)
                    i = i+1   

definedtermdf = pd.DataFrame(definedtermlist)

In [ ]:
srcids = df_clean['identifier'].unique().tolist()
#r = requests.get('https://api-staging.data.niaid.nih.gov/v1/metadata')
#res = json.loads(r.text)
srcnamelist = list(sourcemeta.keys())
templist = []
for eachsrc in srcnamelist:
    if sourcemeta[eachsrc]['sourceInfo']['identifier'] in srcids:
        if 'temporalCoverage' in list(sourcemeta[eachsrc]['sourceInfo'].keys()):
            tempobject = sourcemeta[eachsrc]['sourceInfo']['temporalCoverage']
            if isinstance(tempobject, dict): 
                tempdict = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                              'startDate': tempobject['startDate'],
                              'temporalType': tempobject['temporalType']}
                try:
                    tempdict['endDate']= tempobject['endDate']
                    templist.append(tempdict)
                except:
                    templist.append(tempdict)
            elif isinstance(tempobject, list):
                i=0
                while i < len(tempobject):
                    tempdict = {'url': sourcemeta[eachsrc]['sourceInfo']['url'],
                                  'temporalType': tempobject[i]['temporalType']}
                    try:
                        tempdict['startDate']= tempobject[i]['startDate']
                    except:
                        print(eachsrc,' has no startDate')
                    try:
                        tempdict['endDate']= tempobject[i]['endDate']
                        templist.append(tempdict)
                    except:
                        templist.append(tempdict)
                    i = i+1

tempdf = pd.DataFrame(templist)

In [ ]:
print(tempdf.head(n=2))
print(collectiondf.head(n=2))
print(definedtermdf.head(n=2))

In [ ]:
print(filepath)

In [ ]:
with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:  
    collectiondf.to_excel(writer, sheet_name='collectionSize', index=False)
    definedtermdf.to_excel(writer, sheet_name='definedTerms', index=False)
    tempdf.to_excel(writer, sheet_name='spatialTemporalCoverage', index=False)